# **Working with the OpenAI API**

This Jupyter Notebook documents my comprehensive learning notes, key concepts, and practical implementations for the **Working with the OpenAI API** course. It covers the chat completions API, environment setup, model parameters, tokenization with `tiktoken`, output structuring, moderation guards, and multi-turn conversational agents.

## **Chapter 1: The Chat Completions API, Roles, and Parameters**

DataCamp's curriculum covers the core structure of the Chat Completions endpoint, including messages list structuring, role definitions, and parameter fine-tuning.

### **1.1: Core Message Roles**
1.  **`system`**: Sets the behavior, persona, and guardrails for the assistant (e.g., "You are a database auditor.").
2.  **`user`**: The prompt or query sent by the human user.
3.  **`assistant`**: The model's responses. We append these to maintain history.

### **1.2: Fine-Tuning API Parameters**
- `model`: Specifies the model to call (e.g., `gpt-3.5-turbo`, `gpt-4o`).
- `max_tokens`: Restricts the maximum output length to control costs and latency.
- `temperature`: Scales the randomness of token choices (T between 0 and 2).
- `top_p`: Nucleus sampling; controls diversity by sampling from top percentage of cumulative probability.
- `presence_penalty`: Encourages model to talk about new topics (from -2.0 to 2.0).
- `frequency_penalty`: Penalizes repetitive words (from -2.0 to 2.0).
- `n`: Number of completion choices to return for a single prompt.

In [ ]:
# OpenAI Client Setup & Completions with diverse parameters
# from openai import OpenAI
# client = OpenAI(api_key='your-api-key')

print('OpenAI Client Initialized with default settings.')

# Sample structure of a completions request
completion_args = {
    'model': 'gpt-3.5-turbo',
    'messages': [
        {'role': 'system', 'content': 'You are a creative writer.'},
        {'role': 'user', 'content': 'Write a title for a scifi novel.'}
    ],
    'temperature': 0.8,      # Moderately creative
    'top_p': 0.9,            # Nucleus sampling
    'frequency_penalty': 0.5,# Discourage repeating words
    'presence_penalty': 0.3, # Encourage novel topics
    'n': 2                   # Generate 2 choices
}
print('Prompt args prepared:', completion_args)

### **1.3: Temperature and Softmax Probability Mechanics**

The mathematical foundation of token sampling utilizes logit values $z_i$ scaled by Temperature $T$:
$$P(w_i \mid w_{<i}) = \text{Softmax}\left(\frac{z_i}{T}ight) = \frac{e^{z_i / T}}{\sum_{j} e^{z_j / T}}$$

- **Low Temperature ($T \to 0$):** High precision, deterministic token selection.
- **High Temperature ($T \to 2$):** Uniform probability across all words, leading to chaos and creativity.

## **Chapter 2: Text Generation, Structuring Output, and Tiktoken**

### **2.1: Tokenization with `tiktoken`**
OpenAI utilizes sub-word tokenization. To check how many tokens a prompt will consume prior to calling the API, we use the `tiktoken` library.

In [ ]:
import tiktoken

prompt_text = 'Large Language Models process token structures.'

# Get the standard encoding for gpt-3.5-turbo / gpt-4
encoding = tiktoken.encoding_for_model('gpt-3.5-turbo')

# Encode the text to token IDs
token_ids = encoding.encode(prompt_text)
print('Token IDs:', token_ids)
print('Token Count:', len(token_ids))

# Decode back to verify
decoded_text = encoding.decode(token_ids)
print('Decoded Text:', decoded_text)

### **2.2: Structuring Output (JSON Mode)**
Forcing the model to output a structured JSON schema simplifies software integrations.

In [ ]:
import json

# Enforcing JSON output
json_prompt = {
    'model': 'gpt-4o',
    'response_format': {'type': 'json_object'},
    'messages': [
        {'role': 'system', 'content': 'You are a JSON formatter. Output a JSON object containing key "summary" and key "words_count".'},
        {'role': 'user', 'content': 'Pandas is an open-source data analysis library for Python.'}
    ]
}

# Parsing simulation
raw_response = '{"summary": "Pandas is a Python library for data analysis.", "words_count": 8}'
parsed = json.loads(raw_response)
print('Parsed JSON summary:', parsed['summary'])

## **Chapter 3: Moderation, Guards, and Safe Generation**

### **3.1: OpenAI Moderation Endpoint**
Scans texts across categories to detect violations of usage policy.

In [ ]:
# Simulated Moderation response parsing
moderation_results = {
    "flagged": True,
    "categories": {
        "hate": False,
        "hate/threatening": False,
        "harassment": True,
        "self-harm": False,
        "sexual": False,
        "violence": True
    },
    "category_scores": {
        "harassment": 0.89,
        "violence": 0.95
    }
}

if moderation_results["flagged"]:
    print("Policy Violation Detected!")
    for cat, status in moderation_results["categories"].items():
        if status:
            score = moderation_results["category_scores"].get(cat, 0.0)
            print(f"- Category: {cat} (Score: {score:.2f})")

## **Chapter 4: Text Summarization & Context Processing**

We can instruct the model to produce custom summary formats (e.g. single-sentence summaries, bulleted highlights, or specific tones) while managing context length limits.

In [ ]:
sample_article = (
    'Python is dynamically typed and garbage-collected. '
    'It supports multiple programming paradigms, including structured, object-oriented, '
    'and functional programming.'
)

# Prompt construction targeting specific output formats
summarize_messages = [
    {'role': 'system', 'content': 'Summarize the input article in exactly three bullet points in a professional tone.'},
    {'role': 'user', 'content': sample_article}
]
print('Summarization Prompt Context:\n', summarize_messages)

## **Chapter 5: Multi-Turn Conversation Memory**

To build conversation pipelines, we implement a chat loop that maintains list structures of prior conversation blocks.

In [ ]:
conversation = [
    {'role': 'system', 'content': 'You are a customer support agent.'}
]

def chat(user_msg):
    conversation.append({'role': 'user', 'content': user_msg})
    # Simulated completion response
    assistant_reply = f'Thanks for asking! You said "{user_msg}". How else can I assist?'
    conversation.append({'role': 'assistant', 'content': assistant_reply})
    return assistant_reply

print(chat('Where is my package?'))
print(chat('Can I get a refund?'))